# ============================================================
# BENCHMARK METHOD 4: RandAR Baseline (Table 3 Reproduction)
# Decoder-only Autoregressive Visual Generation in Random Orders (CVPR 2025 Oral)
# Flat Vocabulary (N = 16,384) | 10-Class ImageNet Evaluation
# ============================================================


In [ ]:
# ============================================================
# CELL 1 — Thiết lập môi trường & Clone các Repositories cần thiết
# ============================================================
import subprocess, os, sys, shutil, torch

WORK_DIR          = '/kaggle/working'
RANDAR_DIR        = f'{WORK_DIR}/RandAR'
LLAMA_DIR         = f'{WORK_DIR}/LlamaGen'
BENCHMARK_DIR     = f'{WORK_DIR}/Benchmark'
WEIGHTS_DIR       = f'{WORK_DIR}/weights'
CODES_DIR         = f'{WORK_DIR}/codes'
CKPT_RANDAR_DIR   = f'{WORK_DIR}/checkpoints_randar'

os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(CKPT_RANDAR_DIR, exist_ok=True)
os.makedirs(CODES_DIR, exist_ok=True)

print('Cài đặt dependencies cho RandAR & LlamaGen Evaluator...')
subprocess.run('pip install -q omegaconf timm safetensors einops huggingface_hub pandas tabulate scipy', shell=True)

def clone_or_pull(url, target):
    if not os.path.exists(target):
        print(f'>>> git clone {url} {target}')
        subprocess.run(f'git clone {url} {target}', shell=True, check=True)
    else:
        print(f'>>> {target} đã tồn tại, git pull cập nhật...')
        subprocess.run(f'git -C {target} pull', shell=True)

clone_or_pull('https://github.com/ziqipang/RandAR.git', RANDAR_DIR)
clone_or_pull('https://github.com/FoundationVision/LlamaGen.git', LLAMA_DIR)
clone_or_pull('https://github.com/nta2112/Masce-baseline-benchmark.git', BENCHMARK_DIR)

print('=' * 60)
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
print(f'Số lượng GPU    : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'   GPU {i} : {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)')
print('=' * 60)


In [ ]:
# ============================================================
# CELL 2 — Tải Weights Pretrained của RandAR & VQ-VAE (LlamaGen Tokenizer)
# ============================================================
from huggingface_hub import hf_hub_download
import os

# 1. VQ-VAE Tokenizer (Chung chuẩn với LlamaGen: VQ-DS16, codebook 16,384)
VQ_PATH = os.path.join(WEIGHTS_DIR, 'vq_ds16_c2i.pt')
if not os.path.exists(VQ_PATH):
    print('Đang tải Tokenizer vq_ds16_c2i.pt...')
    hf_hub_download(repo_id='FoundationVision/LlamaGen', filename='vq_ds16_c2i.pt', local_dir=WEIGHTS_DIR)
    print('✅ Đã tải: vq_ds16_c2i.pt')
else:
    print('✅ Tokenizer vq_ds16_c2i.pt đã có sẵn.')

# 2. Checkpoint Pretrained của RandAR-L (0.3B ~ 343M params)
RANDAR_CKPT_NAME = 'randar_0.3b_llamagen_360k_bs_1024_lr_0.0004.safetensors'
RANDAR_CKPT_PATH = os.path.join(WEIGHTS_DIR, RANDAR_CKPT_NAME)
if not os.path.exists(RANDAR_CKPT_PATH):
    print(f'Đang tải Pretrained RandAR-L weights ({RANDAR_CKPT_NAME})...')
    hf_hub_download(repo_id='ziqipang/RandAR', filename=RANDAR_CKPT_NAME, local_dir=WEIGHTS_DIR)
    print(f'✅ Đã tải: {RANDAR_CKPT_NAME}')
else:
    print(f'✅ RandAR checkpoint đã có sẵn: {RANDAR_CKPT_PATH}')


In [ ]:
# ============================================================
# CELL 3 — Chuẩn bị Dữ liệu Huấn luyện & Reference Batch 10-Class
# ĐÚNG 10 CLASS (Classes 0..9) x 1,000 ảnh = 10,000 ảnh chuẩn
# ============================================================
import os, glob, io, tarfile, time, shutil, subprocess
import numpy as np, torch
from pathlib import Path
from huggingface_hub import snapshot_download

REF_NPZ = f'{WORK_DIR}/ref_10class_balanced_1000.npz'
labels_path = f'{CODES_DIR}/labels.npy'

# Tìm kiếm Reference Batch có sẵn trong Kaggle input để tiết kiệm thời gian
REF_INPUT_CANDIDATES = [
    REF_NPZ,
    '/kaggle/input/ref-10class-balanced-1000/ref_10class_balanced_1000.npz',
] + glob.glob('/kaggle/input/**/ref_10class_balanced_1000.npz', recursive=True)

existing_ref = next((p for p in REF_INPUT_CANDIDATES if os.path.exists(p) and os.path.getsize(p) > 1000000), None)
if existing_ref and not os.path.exists(REF_NPZ):
    print(f'✅ Tìm thấy Reference Batch có sẵn trong input: {existing_ref}')
    shutil.copy(existing_ref, REF_NPZ)
    print(f'   -> Đã sao chép sang: {REF_NPZ} ({os.path.getsize(REF_NPZ)/1e6:.1f} MB)')

# Trích xuất 10,000 samples token codes nếu chưa có
existing_npy = sorted(Path(CODES_DIR).glob('shard_*.npy'))
if not (existing_npy and os.path.exists(labels_path) and len(np.load(labels_path)) == 10000):
    WDS_DIR = f'{WORK_DIR}/wds_cache'
    os.makedirs(WDS_DIR, exist_ok=True)
    os.makedirs(CODES_DIR, exist_ok=True)
    print('Đang tải imagenet-llamagen-cache để lấy đúng 10,000 ảnh huấn luyện (10 class x 1,000 ảnh)...')
    snapshot_download(
        repo_id='Efficient-Large-Model/imagenet-llamagen-cache',
        repo_type='dataset', local_dir=WDS_DIR,
        ignore_patterns=['*.json', '*.yaml', 'README*'],
    )
    tar_files = sorted(Path(WDS_DIR).glob('**/*.tar'))
    if not tar_files:
        tar_files = sorted(Path('/kaggle/input').glob('**/*.tar'))
    assert len(tar_files) > 0, 'Không tìm thấy file tar của dataset!'
    tar_path = tar_files[0]
    print(f'Đang trích xuất 10 class từ {tar_path.name}...')

    TARGET_SYNSETS = {
        'n01440764': 0, 'n01443537': 1, 'n01484850': 2, 'n01491361': 3, 'n01494475': 4,
        'n01496331': 5, 'n01498041': 6, 'n01514668': 7, 'n01514859': 8, 'n01518878': 9,
    }
    MAX_PER_CLASS = 1000
    class_counts = {c: 0 for c in range(10)}
    shard_codes, all_labels = [], []
    shard_idx, total_imgs = 0, 0
    SHARD_SIZE = 512

    with tarfile.open(tar_path, 'r:*') as tf:
        for member in tf:
            if not member.name.endswith('.npz'): continue
            parts = member.name.split('/')
            if len(parts) < 2: continue
            synset = parts[-2]
            label = TARGET_SYNSETS.get(synset, -1)
            if label == -1 or class_counts[label] >= MAX_PER_CLASS: continue

            f = tf.extractfile(member)
            if f is None: continue
            try:
                npz = np.load(io.BytesIO(f.read()))
                codes = npz['indices'].flatten().astype(np.int32)
            except Exception: continue
            if len(codes) != 256: continue

            shard_codes.append(codes)
            all_labels.append(label)
            class_counts[label] += 1
            total_imgs += 1

            if len(shard_codes) >= SHARD_SIZE:
                np.save(f'{CODES_DIR}/shard_{shard_idx:05d}.npy', np.array(shard_codes, dtype=np.int32))
                shard_codes = []; shard_idx += 1
            if total_imgs >= 10000: break

    if shard_codes:
        np.save(f'{CODES_DIR}/shard_{shard_idx:05d}.npy', np.array(shard_codes, dtype=np.int32))
    np.save(labels_path, np.array(all_labels, dtype=np.int32))
    shutil.rmtree(WDS_DIR, ignore_errors=True)
    print(f'✅ Đã chuẩn bị 10,000 samples token codes trong {CODES_DIR}!')
else:
    print(f'✅ Đã có sẵn 10,000 samples token codes trong {CODES_DIR}!')

# Tạo Reference Batch (1,000 ảnh thật cân bằng 10 class) nếu chưa có
if not os.path.exists(REF_NPZ):
    print('Đang tạo Reference Batch 10-Class CÂN BẰNG (100 ảnh/class x 10 class = 1,000 ảnh)...')
    if LLAMA_DIR not in sys.path: sys.path.insert(0, LLAMA_DIR)
    from tokenizer.tokenizer_image.vq_model import VQ_models
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

    vq = VQ_models['VQ-16'](codebook_size=16384, codebook_embed_dim=8)
    vq_ckpt = os.path.join(WEIGHTS_DIR, 'vq_ds16_c2i.pt')
    st = torch.load(vq_ckpt, map_location='cpu')
    vq.load_state_dict(st.get('model', st))
    vq.to(device).eval()

    shards = sorted(Path(CODES_DIR).glob('shard_*.npy'))
    all_codes = np.concatenate([np.load(f) for f in shards], axis=0)
    all_labels = np.load(labels_path)

    ref_indices = []
    for c in range(10):
        c_idxs = np.where(all_labels == c)[0][:100]
        ref_indices.extend(c_idxs)
    ref_indices = np.array(ref_indices)
    ref_codes = all_codes[ref_indices]

    ref_imgs = []
    bs = 20
    with torch.no_grad():
        for i in range(0, len(ref_codes), bs):
            b_c = torch.tensor(ref_codes[i:i+bs], device=device, dtype=torch.long)
            imgs = vq.decode_code(b_c, [len(b_c), 8, 16, 16])
            imgs = ((imgs.clamp(-1, 1) + 1.0) / 2.0 * 255.0).permute(0, 2, 3, 1).to(torch.uint8).cpu().numpy()
            ref_imgs.append(imgs)

    ref_arr = np.concatenate(ref_imgs, axis=0)
    np.savez_compressed(REF_NPZ, arr_0=ref_arr)
    print(f'✅ Reference Batch hoàn thành: {REF_NPZ} ({ref_arr.shape})')
    del vq; torch.cuda.empty_cache()
else:
    print(f'✅ Reference Batch đã có sẵn: {REF_NPZ}')


In [ ]:
# ============================================================
# CELL 4 — HUẤN LUYỆN 20 EPOCHS CHO RANDAR BASELINE (N=16,384)
# Tự động phát hiện checkpoint nếu đã có sẵn
# ============================================================
import subprocess, os, glob, torch

RANDAR_FINAL_CKPT = f'{CKPT_RANDAR_DIR}/ckpt_randar_10class_final.pt'

CKPT_CANDIDATES = [
    RANDAR_FINAL_CKPT,
    '/kaggle/input/**/ckpt_randar_10class_final.pt',
] + glob.glob('/kaggle/input/**/ckpt_randar_10class*.pt', recursive=True)

EXISTING_CKPT = next((p for p in CKPT_CANDIDATES if os.path.exists(p)), None)
FRESH_TRAIN = True   # True: Huấn luyện đủ 20 Epochs trên 10 class để so sánh công bằng với MASC

if EXISTING_CKPT and not FRESH_TRAIN:
    print('=' * 70)
    print(f'✅ PHÁT HIỆN CHECKPOINT RANDAR 10-CLASS CÓ SẴN: {EXISTING_CKPT}')
    print('=' * 70)
    print('⏩ BỎ QUA QUÁ TRÌNH HUẤN LUYỆN 20 EPOCHS!')
    print('👉 Chuyển thẳng sang CELL 5 để sinh ảnh và CELL 6 để đánh giá.')
else:
    print('Bắt đầu HUẤN LUYỆN 20 Epochs cho RandAR Baseline (Random-order AR, N=16,384) trên 2x GPU T4...')
    TRAIN_SCRIPT = f'{BENCHMARK_DIR}/baselines/04_randar/train_randar.py'
    if not os.path.exists(TRAIN_SCRIPT):
        TRAIN_SCRIPT = f'{WORK_DIR}/train_randar.py'

    cmd = [
        'torchrun', '--nproc_per_node=2', '--master_port=29520',
        TRAIN_SCRIPT,
        '--randar-repo',   RANDAR_DIR,
        '--vq-ckpt',       f'{WEIGHTS_DIR}/vq_ds16_c2i.pt',
        '--pretrained',    RANDAR_CKPT_PATH,
        '--codes-dir',     CODES_DIR,
        '--epochs',        '20.0',
        '--batch-size',    '32',
        '--grad-accum',    '1',
        '--lr',            '1e-4',
        '--min-lr',        '1e-6',
        '--out-dir',       CKPT_RANDAR_DIR,
        '--save-every',    '5',
        '--log-every',     '50'
    ]
    print(' '.join(cmd))
    print('=' * 70)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode == 0:
        print('\n✅ HUẤN LUYỆN 20 EPOCHS CHO RANDAR BASELINE HOÀN TẤT THÀNH CÔNG!')
    else:
        print('\n❌ Lỗi trong quá trình huấn luyện!')


In [ ]:
# ============================================================
# CELL 5 — Sinh 1,000 Ảnh Mẫu từ RandAR Baseline (Song song 2x GPU T4)
# Sinh chuẩn 100 ảnh/class cho ĐÚNG 10 CLASS (Classes 0..9)
# ============================================================
import subprocess, os, sys

INFER_SCRIPT = f'{BENCHMARK_DIR}/baselines/04_randar/randar_infer.py'
if not os.path.exists(INFER_SCRIPT):
    INFER_SCRIPT = f'{WORK_DIR}/randar_infer.py'

SAMPLES_OUT = f'{WORK_DIR}/samples_randar_baseline_1000.npz'
CKPT_USE    = f'{CKPT_RANDAR_DIR}/ckpt_randar_10class_final.pt'
if not os.path.exists(CKPT_USE):
    CKPT_USE = RANDAR_CKPT_PATH

cmd = [
    'python', INFER_SCRIPT,
    '--randar-repo',  RANDAR_DIR,
    '--vq-ckpt',      f'{WEIGHTS_DIR}/vq_ds16_c2i.pt',
    '--ckpt',         CKPT_USE,
    '--num-samples',  '1000',
    '--batch-size',   '16',
    '--cfg',          '3.0',
    '--top-k',        '1000',
    '--top-p',        '1.0',
    '--steps',        '88',
    '--out',          SAMPLES_OUT
]

print('Bắt đầu sinh 1,000 ảnh RandAR Baseline trên 2x GPU T4 (10 class cân bằng)...')
print(' '.join(cmd))
print('=' * 70)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode == 0:
    print(f'\n✅ Sinh 1,000 ảnh thành công: {SAMPLES_OUT}')
else:
    print('\n❌ Lỗi trong quá trình sinh ảnh!')


In [ ]:
# ============================================================
# CELL 6 — ĐÁNH GIÁ METRICS: FID, IS, Precision, Recall
# So sánh sòng phẳng trên cùng Reference Batch ref_10class_balanced_1000.npz
# ============================================================
import subprocess, os, sys

SAMPLES_OUT = f'{WORK_DIR}/samples_randar_baseline_1000.npz'
REF_NPZ     = f'{WORK_DIR}/ref_10class_balanced_1000.npz'

eval_cmd = f'CUDA_VISIBLE_DEVICES=1 python {LLAMA_DIR}/evaluations/c2i/evaluator.py {REF_NPZ} {SAMPLES_OUT}'
print('Khởi chạy LlamaGen Evaluator trên GPU 1 (dành trọn VRAM cho TF Inception-v3):')
print(eval_cmd)
print('=' * 70)

proc = subprocess.Popen(eval_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('\n✅ HOÀN TẤT ĐÁNH GIÁ RANDAR BASELINE TRÊN ĐÚNG 10 CLASS!')


In [ ]:
# ============================================================
# CELL 7 — BẢNG ĐỐI SÁNH TOÀN DIỆN (TABLE 1 & TABLE 3 BENCHMARK)
# ============================================================
import pandas as pd

table_benchmark = [
    {
        'Phương pháp': 'Vanilla LlamaGen-L (Baseline Flat)',
        'Kiến trúc': 'Next-Token AR (Raster-scan)',
        'Tokens': '256 (N=16,384)',
        '10-Class FID ↓': '35.24',
        'IS ↑': '8.79',
        'Precision ↑': '0.804',
        'Recall ↑': '0.448'
    },
    {
        'Phương pháp': 'k-Means Clustering Baseline',
        'Kiến trúc': 'Next-Token AR (k=4,096)',
        'Tokens': '256 (k=4,096)',
        '10-Class FID ↓': '31.42',
        'IS ↑': '8.85',
        'Precision ↑': '0.705',
        'Recall ↑': '0.533'
    },
    {
        'Phương pháp': 'LlamaGen-L + MASC + UPipe (Ours)',
        'Kiến trúc': 'Next-Token AR + MASC',
        'Tokens': '256 (k=4,096)',
        '10-Class FID ↓': '31.62',
        'IS ↑': '8.92',
        'Precision ↑': '0.744',
        'Recall ↑': '0.539'
    },
    {
        'Phương pháp': 'VAR-d16 Baseline (Tian et al. 2024)',
        'Kiến trúc': 'Next-Scale AR (2D Pyramid)',
        'Tokens': '680 (V=4,096)',
        '10-Class FID ↓': '30.34',
        'IS ↑': '8.74',
        'Precision ↑': '0.684',
        'Recall ↑': '0.653'
    },
    {
        'Phương pháp': 'RandAR Baseline (Pang et al. 2025)',
        'Kiến trúc': 'Random-order AR (N=16,384)',
        'Tokens': '256 (N=16,384)',
        '10-Class FID ↓': '[Kết quả Cell 6]',
        'IS ↑': '[Kết quả Cell 6]',
        'Precision ↑': '[Kết quả Cell 6]',
        'Recall ↑': '[Kết quả Cell 6]'
    }
]

df = pd.DataFrame(table_benchmark)
print(df.to_string(index=False))
